# 01: Dataset & Agent

**Thomson Reuters | Agentic AI Evaluation Bootcamp**

This notebook introduces the two foundational building blocks of our evaluation:

- **DeepSearchQA** — the benchmark dataset: 896 multi-hop research questions across 17 categories
- **KnowledgeGroundedAgent** — the PlanReAct agent with 5 tools, running on Gemini

## What You'll Do

1. Explore the DeepSearchQA dataset — browse categories, question types, and examples
2. Run the agent on a single Finance & Economics question
3. Inspect the full `AgentResponse` — plan, tool calls, sources, and reasoning chain
4. Verify that Langfuse tracing is capturing the agent's trace

## Prerequisites

- All keys set in `~/.env`: `GOOGLE_API_KEY`, `LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY`, `LANGFUSE_HOST`
- Dependencies installed: `uv sync`

## 1. Setup

In [20]:
import os
from pathlib import Path

from aieng.agent_evals.knowledge_qa import DeepSearchQADataset, KnowledgeGroundedAgent
from aieng.agent_evals.knowledge_qa.system_instructions import build_system_instructions
from aieng.agent_evals.langfuse import init_tracing
from dotenv import load_dotenv
from rich.console import Console
from rich.markdown import Markdown
from rich.panel import Panel
from rich.table import Table

# Set working directory to repo root
if Path('').absolute().name == 'eval-agents':
    print(f'Working directory: {Path("").absolute()}')
else:
    os.chdir(Path('').absolute().parent.parent)
    print(f'Working directory set to: {Path("").absolute()}')

load_dotenv(verbose=True)
init_tracing()  # Connects to Langfuse — traces will appear at us.cloud.langfuse.com
console = Console(width=100)

# Category we'll focus on throughout the bootcamp
CATEGORY = 'History'

console.print(Panel(
    f'[bold]Category focus:[/bold] {CATEGORY}\n'
    '[dim]Change CATEGORY above to explore other domains[/dim]',
    title='TR Knowledge QA — Notebook 01',
    border_style='cyan'
))

Working directory: /home/coder/eval-agents


╭───────────────────────────────── TR Knowledge QA — Notebook 01 ──────────────────────────────────╮
│ Category focus: History                                                                          │
│ Change CATEGORY above to explore other domains                                                   │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯

## 2. The DeepSearchQA Dataset

[DeepSearchQA](https://www.kaggle.com/datasets/deepmind/deepsearchqa) is a benchmark from Google DeepMind
for evaluating deep research agents. It contains **896 questions** across **17 categories** that require
multi-step web search and reasoning — not recall from training data.

Each question is a **causal chain task**: the agent must search, fetch real sources, and verify facts.

| Answer Type | Description | Example |
|-------------|-------------|--------|
| **Single Answer** | One specific value | A date, a number, a name |
| **Set Answer** | Multiple required items | A list of companies, a set of policy changes |

In [21]:
dataset = DeepSearchQADataset()

console.print(f'Total examples: [cyan]{len(dataset)}[/cyan]')
console.print(f'Total categories: [cyan]{len(dataset.get_categories())}[/cyan]')

2026-03-24 11:49:06,385 INFO aieng.agent_evals.knowledge_qa.data.deepsearchqa: Downloading DeepSearchQA dataset...
2026-03-24 11:49:07,125 INFO aieng.agent_evals.knowledge_qa.data.deepsearchqa: Dropped 4 examples with missing answers
2026-03-24 11:49:07,127 INFO aieng.agent_evals.knowledge_qa.data.deepsearchqa: Loaded 896 examples from DeepSearchQA


Total examples: 896

Total categories: 17

In [22]:
# Distribution across all 17 categories
cat_table = Table(title='Dataset by Category')
cat_table.add_column('Category', style='cyan')
cat_table.add_column('Total', style='white', justify='right')
cat_table.add_column('Single Answer', style='dim', justify='right')
cat_table.add_column('Set Answer', style='dim', justify='right')

for cat in sorted(dataset.get_categories()):
    examples = dataset.get_by_category(cat)
    single = sum(1 for e in examples if e.answer_type == 'Single Answer')
    set_ans = len(examples) - single
    cat_table.add_row(cat, str(len(examples)), str(single), str(set_ans))

console.print(cat_table)

                     Dataset by Category                      
┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┓
┃ Category              ┃ Total ┃ Single Answer ┃ Set Answer ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━┩
│ Arts                  │    26 │            10 │         16 │
│ Arts & Entertainment  │     1 │             0 │          1 │
│ Biology               │     2 │             0 │          2 │
│ Current Events        │     3 │             0 │          3 │
│ Education             │    94 │            38 │         56 │
│ Finance & Economics   │   132 │            56 │         76 │
│ Geography             │    95 │            32 │         63 │
│ Health                │    92 │            34 │         58 │
│ History               │    44 │            13 │         31 │
│ Linguistics           │     1 │             1 │          0 │
│ Media & Entertainment │    27 │            11 │         16 │
│ Other                 │    65 │            22 │         43 │
│ Politics & Government │   148 │            37 │        111 │
│ Science               │    89 │            30 │         59 │
│ Sports                │    19 │             6 │         13 │
│ Technology            │    22 │            11 │         11 │
│ Travel                │    36 │            15 │         21 │
└───────────────────────┴───────┴───────────────┴────────────┘

In [25]:
# Browse Finance & Economics examples
question_examples = dataset.get_by_category(CATEGORY)
console.print(f'[bold]{CATEGORY}[/bold]: [cyan]{len(question_examples)}[/cyan] examples\n')

browse_table = Table(title=f'{CATEGORY} — First 5 Examples')
browse_table.add_column('ID', style='dim', width=6)
browse_table.add_column('Answer Type', style='cyan', width=15)
browse_table.add_column('Question', style='white')

for ex in question_examples[:5]:
    q = ex.problem[:80] + '...' if len(ex.problem) > 80 else ex.problem
    browse_table.add_row(str(ex.example_id), ex.answer_type, q)

console.print(browse_table)

History: 44 examples

                                     History — First 5 Examples                                     
┏━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ ID     ┃ Answer Type     ┃ Question                                                              ┃
┡━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 13     │ Single Answer   │ In the year that featured a volcanic eruption in Iceland causing      │
│        │                 │ widespread air-...                                                    │
│ 70     │ Set Answer      │ Please provide me with a list of fires from the San Francisco         │
│        │                 │ database of fires ...                                                 │
│ 109    │ Single Answer   │ Who is the Associate Editor of the SBL Study Bible who is a graduate  │
│        │                 │ of the Univ...                                                        │
│ 122    │ Set Answer      │ Using information from the Council on Tall Buildings and Urban        │
│        │                 │ Habitat rankings ...                                                  │
│ 123    │ Set Answer      │ According to the Social Security Administration, what are the         │
│        │                 │ identical names th...                                                 │
└────────┴─────────────────┴───────────────────────────────────────────────────────────────────────┘

In [26]:
# Inspect one example in full — this is what the agent will receive as input
example = question_examples[1]

console.print(
    Panel(
        f'[bold]example_id:[/bold]       {example.example_id}\n'
        f'[bold]problem_category:[/bold] {example.problem_category}\n'
        f'[bold]answer_type:[/bold]      {example.answer_type}\n\n'
        f'[bold cyan]Question:[/bold cyan]\n{example.problem}\n\n'
        f'[bold yellow]Ground Truth Answer:[/bold yellow]\n{example.answer}',
        title='DSQAExample',
        border_style='blue',
    )
)

╭────────────────────────────────────────── DSQAExample ───────────────────────────────────────────╮
│ example_id:       70                                                                             │
│ problem_category: History                                                                        │
│ answer_type:      Set Answer                                                                     │
│                                                                                                  │
│ Question:                                                                                        │
│ Please provide me with a list of fires from the San Francisco database of fires up to May 2024   │
│ that involved more than 1000 suppression units and had the date of the incident after 2010.      │
│ Please give the incident number of these fires.                                                  │
│                                                                                                  │
│ Ground Truth Answer:                                                                             │
│ 11120831, 14018336                                                                               │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯

## 3. The Agent's Tools

The `KnowledgeGroundedAgent` has five tools that form a research workflow:

| Tool | Purpose | When Used |
|------|---------|----------|
| `google_search` | Find relevant URLs | First step for any sub-question |
| `web_fetch` | Read full web pages / PDFs | Verify facts from the actual source |
| `fetch_file` | Download CSV, XLSX, JSON | When the answer is in structured data |
| `grep_file` | Search within a downloaded file | Locate a specific value in a large file |
| `read_file` | Read sections of a downloaded file | Inspect a specific part of a file |

The system instructions enforce: **Search → Fetch → Verify → Answer** (never answer from snippets alone).

In [27]:
instructions = build_system_instructions()
console.print(
    Panel(
        Markdown(instructions),
        title='Agent System Instructions',
        border_style='blue',
        padding=(1, 2),
    )
)

╭─────────────────────────────────── Agent System Instructions ────────────────────────────────────╮
│                                                                                                  │
│  You are a research assistant that finds accurate answers by exploring sources and verifying     │
│  facts.                                                                                          │
│                                                                                                  │
│  Today's date: March 24, 2026                                                                    │
│                                                                                                  │
│  Tools                                                                                           │
│                                                                                                  │
│  google_search: Find URLs related to a topic. Search results include brief snippets—use these    │
│  to identify promising sources, then fetch pages for complete information.                       │
│                                                                                                  │
│  web_fetch: Read the full content of a web page. Use this to verify facts and find detailed      │
│  information.                                                                                    │
│                                                                                                  │
│  fetch_file: Download data files (CSV, XLSX, JSON) for structured data like statistics or        │
│  datasets.                                                                                       │
│                                                                                                  │
│  grep_file: Search within a downloaded file to locate specific information.                      │
│                                                                                                  │
│  read_file: Read sections of a downloaded file to examine data in detail.                        │
│                                                                                                  │
│  Search Strategy                                                                                 │
│                                                                                                  │
│  Search for the answer, not just context. If a question asks "what are the three categories of   │
│  X?", search for those categories directly rather than first identifying what X is and then      │
│  searching within X.                                                                             │
│                                                                                                  │
│  Keep key terms together. Include the core question terms in your search query. A search         │
│  combining the key concepts often finds the answer more directly than breaking it into separate  │
│  searches.                                                                                       │
│                                                                                                  │
│  Avoid premature commitment. Don't lock onto an interpretation early. If you assume something    │
│  is "Game A" and search for answers within "Game A", you may miss the correct answer if your     │
│  assumption was wrong. Stay open until you have confirming evidence.                             │
│                                                                                                  │
│  Adapting Your Plan                                                                              │
│                                                                                                  │
│  If your initial approach doesn't yield the needed information:                                  │
│                                                            

## 4. Run the Agent on a Single Question

We run the agent with `enable_planning=True` — this activates the **PlanReAct** architecture:
1. The planner creates an explicit multi-step research plan
2. The worker executes each step in a ReAct loop (reason → tool → observe → repeat)

> **Note:** This takes 30–90 seconds. The agent is doing real web searches.

In [28]:
# Pick the first Finance & Economics question
question = question_examples[0].problem
ground_truth = question_examples[0].answer

console.print(Panel(
    f'[bold cyan]Question:[/bold cyan]\n{question}\n\n'
    f'[bold yellow]Ground Truth:[/bold yellow]\n{ground_truth}',
    title='Running Agent On...',
    border_style='yellow'
))

agent = KnowledgeGroundedAgent(enable_planning=True)
response = await agent.answer_async(question)

console.print('[green]✓[/green] Agent finished!')

╭────────────────────────────────────── Running Agent On... ───────────────────────────────────────╮
│ Question:                                                                                        │
│ In the year that featured a volcanic eruption in Iceland causing widespread air-travel           │
│ disruption across Europe and the northern hemisphere, a historic weather event led to severe     │
│ flooding in England. During this weather event, there is an area that recorded the highest daily │
│ rainfall total and also has a castle. The castle is protected by a drawbridge, does the main     │
│ gatehouse entrance of this castle predominantly face north, south, east, or west?                │
│                                                                                                  │
│ Ground Truth:                                                                                    │
│ West                                                                                             │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯

2026-03-24 11:50:00,073 INFO aieng.agent_evals.knowledge_qa.agent: Answering question: In the year that featured a volcanic eruption in Iceland causing widespread air-travel disruption ac...


/home/coder/eval-agents/.venv/lib/python3.12/site-packages/google/adk/models/google_llm.py:172: UserWarning: [EXPERIMENTAL] GeminiContextCacheManager: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  cache_manager = GeminiContextCacheManager(self.api_client)
2026-03-24 11:50:00,248 INFO google_adk.google.adk.models.google_llm: Sending out request, model: gemini-2.5-flash, backend: GoogleLLMVariant.GEMINI_API, stream: False
2026-03-24 11:50:02,044 INFO google_adk.google.adk.models.google_llm: Response received from the model.
2026-03-24 11:50:02,050 INFO aieng.agent_evals.knowledge_qa.agent: Extracted plan with 4 steps
2026-03-24 11:50:02,051 INFO aieng.agent_evals.knowledge_qa.agent: Tool call: google_search({'query': 'volcanic eruption Iceland air travel disruption Europe northern hemisphere year'})
2026-03-24 11:50:02,240 INFO google_genai.models: AFC is enabled with max remote calls: 10.
202

✓ Agent finished!

## 5. Inspect the AgentResponse

The `AgentResponse` object contains everything the agent produced:

| Field | What it contains |
|-------|------------------|
| `text` | The final answer |
| `plan` | The research plan created before execution |
| `tool_calls` | Every tool call made, with inputs and outputs |
| `sources` | URLs cited in the answer |
| `reasoning_chain` | Step-by-step agent reasoning |
| `total_duration_ms` | Wall-clock time in milliseconds |

In [29]:
# Final answer
console.print(Panel(
    f'[bold]Agent Answer:[/bold]\n{response.text}\n\n'
    f'[bold yellow]Ground Truth:[/bold yellow]\n{ground_truth}\n\n'
    f'[dim]Duration: {response.total_duration_ms / 1000:.1f}s[/dim]',
    title='Agent Response',
    border_style='green'
))

╭───────────────────────────────────────── Agent Response ─────────────────────────────────────────╮
│ Agent Answer:                                                                                    │
│ ANSWER: The main gatehouse entrance of Warblington Castle predominantly faces south.             │
│ SOURCES:                                                                                         │
│ - https://en.wikipedia.org/wiki/Warblington_Castle                                               │
│ - https://historicengland.org.uk/listing/the-list/list-entry/1154484                             │
│ - https://www.gatehouse-gazetteer.info/English%20sites/1299.html                                 │
│ REASONING: The year of the volcanic eruption in Iceland (Eyjafjallajökull) causing widespread    │
│ air-travel disruption was 2010. In November 2010, severe flooding occurred in England,           │
│ specifically around Emsworth in Hampshire, which experienced significant daily rainfall.         │
│ Warblington Castle is located near Emsworth. According to multiple sources, the main gatehouse   │
│ entrance of Warblington Castle is oriented towards the south, with a gateway tower as part of    │
│ the south wall and a south-east octagonal stair turret.                                          │
│                                                                                                  │
│ Ground Truth:                                                                                    │
│ West                                                                                             │
│                                                                                                  │
│ Duration: 44.6s                                                                                  │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯

In [ ]:
question = question_examples[5].problem
ground_truth = question_examples[5].answer
question

'Which municipalities in the Vancouver Lower Mainland saw an increase specifically in passenger vehicle population by 10,000 vehicles or more between 2020 and 2023 according to ICBC Vehicle Population data?'

In [17]:
console.print(Panel(
    f'[bold cyan]Question:[/bold cyan]\n{question}\n\n'
    f'[bold yellow]Ground Truth:[/bold yellow]\n{ground_truth}',
    title='Running Agent On...',
    border_style='yellow'
))

agent = KnowledgeGroundedAgent(enable_planning=True)
response = await agent.answer_async(question)

console.print('[green]✓[/green] Agent finished!')

╭────────────────────────────────────── Running Agent On... ───────────────────────────────────────╮
│ Question:                                                                                        │
│ Which municipalities in the Vancouver Lower Mainland saw an increase specifically in passenger   │
│ vehicle population by 10,000 vehicles or more between 2020 and 2023 according to ICBC Vehicle    │
│ Population data?                                                                                 │
│                                                                                                  │
│ Ground Truth:                                                                                    │
│ Surrey, Burnaby, Richmond, Langley                                                               │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯

2026-03-24 11:46:27,072 INFO aieng.agent_evals.knowledge_qa.agent: Answering question: Which municipalities in the Vancouver Lower Mainland saw an increase specifically in passenger vehic...
2026-03-24 11:46:27,203 INFO google_adk.google.adk.models.google_llm: Sending out request, model: gemini-2.5-flash, backend: GoogleLLMVariant.GEMINI_API, stream: False
2026-03-24 11:46:29,129 INFO google_adk.google.adk.models.google_llm: Response received from the model.
2026-03-24 11:46:29,135 INFO aieng.agent_evals.knowledge_qa.agent: Extracted plan with 7 steps
2026-03-24 11:46:29,136 INFO aieng.agent_evals.knowledge_qa.agent: Tool call: google_search({'query': 'ICBC Vehicle Population data 2020 2023 Vancouver Lower Mainland municipalities'})
2026-03-24 11:46:29,257 INFO google_genai.models: AFC is enabled with max remote calls: 10.
2026-03-24 11:46:34,888 INFO aieng.agent_evals.knowledge_qa.agent: Tool response: google_search completed
2026-03-24 11:46:35,493 INFO google_adk.google.adk.models.g

✓ Agent finished!

In [19]:
# Final answer
console.print(Panel(
    f'[bold]Agent Answer:[/bold]\n{response.text}\n\n'
    f'[bold yellow]Ground Truth:[/bold yellow]\n{ground_truth}\n\n'
    f'[dim]Duration: {response.total_duration_ms / 1000:.1f}s[/dim]',
    title='Agent Response',
    border_style='green'
))

╭───────────────────────────────────────── Agent Response ─────────────────────────────────────────╮
│ Agent Answer:                                                                                    │
│ I am unable to provide a definitive answer to which municipalities in the Vancouver Lower        │
│ Mainland saw an increase of 10,000 or more in passenger vehicle population between 2020 and      │
│ 2023, according to ICBC Vehicle Population data.                                                 │
│                                                                                                  │
│ My attempts to access this data directly from ICBC's official sources (including their           │
│ statistics page and Tableau Public dashboards) were unsuccessful. The Tableau dashboards, which  │
│ are cited as the primary source for this detailed, municipal-level data, require JavaScript to   │
│ operate and to download data, which my current tools cannot execute.                             │
│                                                                                                  │
│ While I found general trends and aggregated statistics regarding vehicle use and registrations   │
│ in Vancouver and British Columbia during the specified period, these sources did not provide the │
│ specific, granular passenger vehicle population numbers for individual Vancouver Lower Mainland  │
│ municipalities needed to perform the required calculation.                                       │
│                                                                                                  │
│ Ground Truth:                                                                                    │
│ Surrey, Burnaby, Richmond, Langley                                                               │
│                                                                                                  │
│ Duration: 47.2s                                                                                  │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯

In [30]:
# Research plan (PlanReAct)
if response.plan:
    plan = response.plan

    # Show planning reasoning
    if plan.reasoning:
        console.print(Panel(plan.reasoning, title='Planning Reasoning', border_style='dim'))

    # Show steps as a table
    plan_table = Table(title=f'Research Plan ({len(plan.steps)} steps)')
    plan_table.add_column('#', style='dim', width=3)
    plan_table.add_column('Type', style='cyan', width=10)
    plan_table.add_column('Status', style='yellow', width=12)
    plan_table.add_column('Description', style='white')

    for step in plan.steps:
        plan_table.add_row(
            str(step.step_id),
            step.step_type,
            step.status.value,
            step.description[:80] + '...' if len(step.description) > 80 else step.description
        )

    console.print(plan_table)
else:
    console.print('[dim]No plan (planning was disabled)[/dim]')

╭─────────────────────────────────────── Planning Reasoning ───────────────────────────────────────╮
│ 1. Search for the year of a volcanic eruption in Iceland that caused widespread air-travel       │
│ disruption across Europe and the northern hemisphere.                                            │
│ 2. Search for historic weather events and severe flooding in England during that identified      │
│ year, specifically looking for areas with the highest daily rainfall.                            │
│ 3. Identify if the area with the highest daily rainfall has a castle.                            │
│ 4. If a castle is identified, search for information about its main gatehouse entrance           │
│ orientation (north, south, east, or                                                              │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯

                                      Research Plan (4 steps)                                       
┏━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ Type       ┃ Status       ┃ Description                                                    ┃
┡━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ research   │ completed    │ Search for the year of a volcanic eruption in Iceland that     │
│     │            │              │ caused widespread air...                                       │
│ 2   │ research   │ completed    │ Search for historic weather events and severe flooding in      │
│     │            │              │ England during that id...                                      │
│ 3   │ research   │ completed    │ Identify if the area with the highest daily rainfall has a     │
│     │            │              │ castle.                                                        │
│ 4   │ research   │ skipped      │ If a castle is identified, search for information about its    │
│     │            │              │ main gatehouse entra...                                        │
└─────┴────────────┴──────────────┴────────────────────────────────────────────────────────────────┘

In [31]:
# Tool calls — this is what trajectory evaluation will inspect
tool_table = Table(title=f'Tool Calls ({len(response.tool_calls)} total)')
tool_table.add_column('#', style='dim', width=3)
tool_table.add_column('Tool', style='cyan', width=18)
tool_table.add_column('Input (truncated)', style='white')

for i, tc in enumerate(response.tool_calls, 1):
    input_str = str(tc.get('input', ''))[:80] + '...' if len(str(tc.get('input', ''))) > 80 else str(tc.get('input', ''))
    tool_table.add_row(str(i), tc.get('name', '?'), input_str)

console.print(tool_table)

              Tool Calls (3 total)              
┏━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ Tool               ┃ Input (truncated) ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ 1   │ google_search      │                   │
│ 2   │ google_search      │                   │
│ 3   │ google_search      │                   │
└─────┴────────────────────┴───────────────────┘

In [32]:
# Sources cited
if response.sources:
    src_table = Table(title='Sources Cited')
    src_table.add_column('#', style='dim', width=3)
    src_table.add_column('URL', style='blue')
    for i, src in enumerate(response.sources, 1):
        src_table.add_row(str(i), str(src))
    console.print(src_table)
else:
    console.print('[yellow]No sources cited — this will score 0.0 on trace_has_sources[/yellow]')

                                           Sources Cited                                            
┏━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ URL                                                                                        ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ title='wikipedia.org'                                                                      │
│     │ uri='https://en.wikipedia.org/wiki/Air_travel_disruption_after_the_2010_Eyjafjallaj%C3%B6… │
│ 2   │ title='wikipedia.org'                                                                      │
│     │ uri='https://en.wikipedia.org/wiki/2010_eruptions_of_Eyjafjallaj%C3%B6kull'                │
│ 3   │ title='simpleflying.com' uri='https://simpleflying.com/eyjafjallajokull-10-years-on/'      │
│ 4   │ title='ncas.ac.uk'                                                                         │
│     │ uri='https://ncas.ac.uk/eyjafjallajokull-2010-how-an-icelandic-volcano-eruption-closed-eu… │
│ 5   │ title='travelpulse.com'                                                                    │
│     │ uri='https://www.travelpulse.com/news/agents/iceland-volcanic-eruption-severely-disrupts-… │
│ 6   │ title='youtube.com' uri='https://www.youtube.com/watch?v=E89ywAUd-NQ'                      │
│ 7   │ title='nerc.ac.uk' uri='https://nora.nerc.ac.uk/id/eprint/10631/1/hs201008.pdf'            │
│ 8   │ title='theguardian.com'                                                                    │
│     │ uri='https://www.theguardian.com/uk/2010/nov/17/borrowdale-rainfall-heaviest-24-hours'     │
│ 9   │ title='curiousclaire.co.uk'                                                                │
│     │ uri='https://www.curiousclaire.co.uk/best-castles-in-hampshire/'                           │
│ 10  │ title='komoot.com' uri='https://www.komoot.com/guide/365008/castles-in-hampshire'          │
│ 11  │ title='greatbritishlife.co.uk'                                                             │
│     │ uri='https://www.greatbritishlife.co.uk/magazines/hampshire/22571990.21-castles-visit-ham… │
│ 12  │ title='muddystilettos.co.uk'                                                               │
│     │ uri='https://hants.muddystilettos.co.uk/things-to-do/castles-hampshire-isle-of-wight/'     │
│ 13  │ title='komoot.com' uri='https://www.komoot.com/guide/3745663/castles-around-dunton-green'  │
│ 14  │ title='visit-hampshire.co.uk'                                                              │
│     │ uri='https://www.visit-hampshire.co.uk/things-to-do/attractions/stately-homes-and-castles' │
│ 15  │ title='wanderlog.com'                                                                      │
│     │ uri='https://wanderlog.com/list/geoCategory/115628/best-castles-in-and-around-new-forest-… │
│ 16  │ title='greatbritishlife.co.uk'                                                             │
│     │ uri='https://www.greatbritishlife.co.uk/magazines/hampshire/22575740.9-castles-forts-can-… │
│ 17  │ title='wikipedia.org' uri='https://en.wikipedia.org/wiki/Warblington_Castle'               │
│ 18  │ title='historicengland.org.uk'                                                             │
│     │ uri='https://historicengland.org.uk/listing/the-list/list-entry/1154484'                   │
│ 19  │ title='gatehouse-gazetteer.info'                                                           │
│     │ uri='https://www.gatehouse-gazetteer.info/English%20sites/1299.html'                       │
└─────┴────────────────────────────────────────────────────────────────────────────────────────────┘

In [33]:
# Confirm Langfuse received the trace
console.print(Panel(
    '[green]✓[/green] Trace sent to Langfuse.\n\n'
    'Go to [bold]us.cloud.langfuse.com[/bold] → Traces\n'
    'to see the full agent trace with all tool call inputs/outputs.',
    title='Langfuse Tracing',
    border_style='green'
))

╭──────────────────────────────────────── Langfuse Tracing ────────────────────────────────────────╮
│ ✓ Trace sent to Langfuse.                                                                        │
│                                                                                                  │
│ Go to us.cloud.langfuse.com → Traces                                                             │
│ to see the full agent trace with all tool call inputs/outputs.                                   │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯

## Summary

In this notebook you:

1. Loaded the **DeepSearchQA** dataset — 896 questions, 17 categories, ground truth included
2. Explored the **Finance & Economics** category (the focus for Build Day)
3. Ran the **KnowledgeGroundedAgent** on one question and inspected its plan, tool calls, and sources
4. Confirmed **Langfuse** is capturing traces

**Next:** Open `02_evaluation.ipynb` to run the LLM-as-judge and score answers against ground truth.

In [34]:
console.print(Panel(
    '[green]✓[/green] Notebook 01 complete!\n\n'
    '[cyan]Next:[/cyan] Open [bold]02_evaluation.ipynb[/bold]',
    title='Done',
    border_style='green',
))

╭────────────────────────────────────────────── Done ──────────────────────────────────────────────╮
│ ✓ Notebook 01 complete!                                                                          │
│                                                                                                  │
│ Next: Open 02_evaluation.ipynb                                                                   │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯